# Filter out categorised professions by count

1. I combined a list of all professions from debiaswe and 100-years-of-stereotypes.
2. I manually curated them and counted their (relevant) occurences in the overall dataset
    - this involved sometimes ensuring noun-usage and other times whole-wordedness
3. Then I partially manually categorised them into the 43 ILO categories
    - some decisions remain to be made about what to include in the 44th "Unemployed" category

Now I want to keep all the top 5 occupations in each category. 

In [1]:
import json
from pathlib import Path

root_dir = Path.cwd().parent.parent
cat_file = root_dir / "data" / "occupations" / "profession_category_to_professions.json"
count_file = root_dir / "data" / "dolci" / "dolci_profession_counts.json"

with open(cat_file, "r") as f:
    data = json.load(f)
with open(count_file, "r") as f:
    counts = json.load(f)

In [2]:
import pandas as pd

# Create a list to store the data
data_list = []

# Iterate through each category and its professions
for category, professions in data.items():
    # Handle cases where professions might be a string instead of a list
    if isinstance(professions, str):
        professions = [professions]
    
    # For each profession in the category
    for profession in professions:
        # Find the count for this profession
        count = next((item['count'] for item in counts if item['profession'] == profession), 0)
        data_list.append({
            'profession': profession,
            'category': category,
            'count': count
        })

# Create the dataframe
df = pd.DataFrame(data_list)
df.head()

,profession,category,count
0,actor,"legal, social and cultural professionals",139454
1,actress,"legal, social and cultural professionals",2589
2,advocate,"legal, social and cultural professionals",3096
3,anthropologist,"legal, social and cultural professionals",1133
4,archbishop,"legal, social and cultural professionals",140


In [ ]:
# keep top-5 professions by count within each category (count > 500)
df_filtered = df[df["count"] > 500]
df_top5 = (
    df_filtered.sort_values(['category', 'count'], ascending=[True, False])
      .groupby('category', group_keys=False)
      .head(5)
      .reset_index(drop=True)
)

len(df_top5)

119

In [6]:
# save top5 df to json
output_file = root_dir / "data" / "occupations" / "top5_professions_by_category.json"
df_top5.to_json(output_file, orient='records', indent=4)